# Preprocessing Teks Ulasan Produk E-Commerce Berbahasa Indonesia
**Nama:** Mhd Zaki Arkan  
**NIM:** 25146061  
**Dataset:** [dipawidia/ecommerce-product-reviews-sentiment](https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment)

Notebook ini mendokumentasikan tahapan eksplorasi dataset dan *text preprocessing* Bahasa Indonesia yang meliputi **Case Folding**, **Cleaning**, **Tokenisasi**, **Stopword Removal**, dan **Stemming**.

## 1. Instalasi dan Import Pustaka

In [1]:
# Instalasi library yang diperlukan (hapus tanda # jika belum terpasang)
# !pip install -q datasets pandas matplotlib seaborn nltk PySastrawi

import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from datasets import load_dataset
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Semua pustaka berhasil diimpor!")

Semua pustaka berhasil diimpor!

## 2. Memuat Dataset & Menampilkan Informasi Dataset
Memuat dataset `dipawidia/ecommerce-product-reviews-sentiment` dari Hugging Face Hub, lalu menampilkan jumlah baris data, daftar kolom, dan distribusi label sentimen.

In [2]:
# Memuat dataset dari Hugging Face
dataset = load_dataset("dipawidia/ecommerce-product-reviews-sentiment")
print("Struktur Split Dataset:", dataset)

# Menggabungkan seluruh split ke dalam DataFrame pandas
dfs = []
for split_name in dataset.keys():
    temp_df = dataset[split_name].to_pandas()
    temp_df["split"] = split_name
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print(f"\nJumlah Total Data : {len(df):,} baris")
print(f"Daftar Kolom      : {list(df.columns)}")
print("\n5 Baris Pertama Dataset:")
df.head()

In [3]:
# Deteksi otomatis kolom teks ulasan dan kolom label sentimen
text_col = next((c for c in df.columns if any(k in c.lower() for k in ["review", "text", "ulasan", "komentar"])), df.columns[0])
label_col = next((c for c in df.columns if any(k in c.lower() for k in ["sentiment", "label", "sentimen", "class"])), df.columns[-1])

print(f"Kolom Teks Ulasan   : '{text_col}'")
print(f"Kolom Label Sentimen: '{label_col}'")

# Distribusi Label Sentimen
distribusi = df[label_col].value_counts(dropna=False).reset_index()
distribusi.columns = ["Label Sentimen", "Jumlah Data"]
distribusi["Persentase (%)"] = (distribusi["Jumlah Data"] / len(df) * 100).round(2)
print("\nDistribusi Label Sentimen:")
print(distribusi.to_string(index=False))

# Visualisasi Distribusi Label Sentimen
plt.figure(figsize=(7, 4))
sns.barplot(data=distribusi, x="Label Sentimen", y="Jumlah Data", hue="Label Sentimen", palette="viridis", legend=False)
plt.title("Distribusi Label Sentimen Ulasan E-Commerce", fontsize=12, fontweight="bold")
plt.xlabel("Label Sentimen")
plt.ylabel("Jumlah Ulasan")
plt.tight_layout()
plt.show()

## 3. Pipeline Preprocessing Teks Bahasa Indonesia
Pipeline preprocessing terdiri dari lima tahapan berurutan:
1. **Case Folding**: Mengubah seluruh huruf menjadi huruf kecil (`lowercase`).
2. **Cleaning**: Menghapus URL, angka, tanda baca, emoji, dan karakter khusus non-alfabetik.
3. **Tokenisasi**: Memecah string teks menjadi daftar token kata menggunakan `nltk.tokenize.word_tokenize`.
4. **Stopword Removal**: Menghapus kata umum (*stopwords*) Bahasa Indonesia dengan tetap mempertahankan kata negasi (*tidak*, *tanpa*, *kurang*) agar polaritas sentimen tidak berubah.
5. **Stemming**: Mengubah kata berimbuhan menjadi kata dasar menggunakan `PySastrawi` (algoritma Nazief-Adriani).

In [4]:
# Inisialisasi Stopword & Stemmer Bahasa Indonesia (Sastrawi)
stop_factory = StopWordRemoverFactory()
stopwords_id = set(stop_factory.get_stop_words())

# Tambahkan stopword informal e-commerce & pertahankan kata negasi penting
extra_stopwords = {
    "yg", "dg", "dgn", "tp", "nya", "ya", "kok", "sih", "deh", "dong",
    "aja", "saja", "banget", "sekali", "sangat", "pas", "sama", "sini",
    "terima", "kasih", "alhamdulillah", "padahal", "lumayan", "ternyata",
    "bakal", "terus", "jauh", "lebih", "pcs", "dll"
}
negation_words = {"tidak", "bukan", "jangan", "kurang", "tanpa", "belum"}
stopwords_id = (stopwords_id | extra_stopwords) - negation_words

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def case_folding(text: str) -> str:
    return str(text).lower()

def cleaning(text: str) -> str:
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # Hapus URL
    text = re.sub(r'[@#]\w+', ' ', text)                  # Hapus mention/hashtag
    text = re.sub(r'\d+', ' ', text)                      # Hapus angka
    text = re.sub(r'[^a-z\s]', ' ', text)                 # Hapus tanda baca & karakter khusus
    text = re.sub(r'\s+', ' ', text).strip()              # Normalisasi spasi
    return text

def tokenize(text: str) -> list:
    return nltk.tokenize.word_tokenize(text)

def remove_stopwords(tokens: list) -> list:
    return [t for t in tokens if t not in stopwords_id and len(t) > 1]

def stem_tokens(tokens: list) -> list:
    return [stemmer.stem(t) for t in tokens]

def preprocess_pipeline(text: str) -> dict:
    cf = case_folding(text)
    cl = cleaning(cf)
    tk = tokenize(cl)
    sw = remove_stopwords(tk)
    st = stem_tokens(sw)
    return {
        "case_folding": cf,
        "cleaning": cl,
        "tokens": tk,
        "no_stopword": sw,
        "stemmed_tokens": st,
        "clean_text": " ".join(st)
    }

print("Fungsi preprocessing siap digunakan.")

Fungsi preprocessing siap digunakan.

## 4. Menampilkan 10 Contoh Teks Sebelum dan Sesudah Preprocessing

In [5]:
# Mengambil 10 sampel ulasan dan menerapkan seluruh tahapan preprocessing
sample_df = df[[text_col, label_col]].dropna().head(10).reset_index(drop=True).copy()
processed_records = sample_df[text_col].apply(preprocess_pipeline)

sample_df["Setelah_Cleaning"] = processed_records.apply(lambda x: x["cleaning"])
sample_df["Tokenisasi"] = processed_records.apply(lambda x: x["tokens"])
sample_df["Setelah_Stopword_Removal"] = processed_records.apply(lambda x: x["no_stopword"])
sample_df["Sesudah_Preprocessing (Stemmed)"] = processed_records.apply(lambda x: x["clean_text"])

for idx, row in sample_df.iterrows():
    print(f"=== Contoh #{idx + 1} [Label: {row[label_col]}] ===")
    print(f"Sebelum : {row[text_col]}")
    print(f"Sesudah : {row['Sesudah_Preprocessing (Stemmed)']}")
    print("-" * 80)

# Menerapkan preprocessing ke seluruh dataset (hapus tanda # untuk menjalankan)
# df["clean_review"] = df[text_col].astype(str).apply(lambda t: preprocess_pipeline(t)["clean_text"])

## 5. Pengaruh Preprocessing terhadap Kualitas Data Teks
Penerapan *text preprocessing* memberikan dampak signifikan terhadap kualitas data teks sebelum pemodelan analisis sentimen:
1. **Reduksi Noise & Karakter Tidak Relevan**: Penghapusan URL, angka, tanda baca, dan simbol menghilangkan elemen yang tidak membawa informasi sentimen sehingga model tidak mempelajari pola acak (*overfitting*).
2. **Standarisasi Bentuk Kata (Normalisasi Kosakata)**: *Case folding* dan *stemming* menyatukan variasi morfologis (*pengiriman*, *dikirim*, *mengirim* $ightarrow$ `kirim`; *mengecewakan*, *kecewa* $ightarrow$ `kecewa`), sehingga mengurangi dimensi ruang fitur (*vocabulary size*) dan mengatasi masalah *data sparsity*.
3. **Peningkatan Rasio Sinyal terhadap Noise (*Signal-to-Noise Ratio*)**: Penghapusan *stopwords* mengeliminasi kata hubung dan preposisi berfrekuensi tinggi, sehingga pembobotan fitur (seperti TF-IDF atau *word embeddings*) terfokus pada kata-kata pembawa makna sentimen utama.
4. **Efisiensi Komputasi & Stabilitas Model**: Ukuran kosakata yang lebih ringkas mempercepat proses pelatihan model *machine learning* maupun *deep learning* serta meningkatkan konsistensi generalisasi pada data uji baru.